# SPY Credit Put Spread — Research Backtest

**Strategy:** Sell 30Δ / Buy 20Δ SPY put spread | 14 DTE | VIX < 20 filter  
**Period:** January 2020 — January 2026  
**Status:** Research complete — framework validated, regime conditioning identified as next iteration

---

## Hypothesis
Equity index options systematically overprice downside risk relative to realized outcomes.
This creates a structural edge for short premium strategies — the volatility risk premium (VRP)
embedded in SPY puts can be harvested by selling spreads in low-volatility regimes.

A VIX level filter below 20 targets entries during calm, low-vol environments where 
short premium historically performs best.

## Research Question
Does a VIX level filter alone produce consistent edge across different market regimes, 
or does performance depend on the broader directional and volatility environment?

## Key Findings
- **176 trades** over 6 years | **80.1% win rate** | **$1,510 total P&L**
- Strategy performs well in low-vol, upward-trending regimes (2021, late 2025)
- Underperforms during sustained directional selloffs — 2022 Fed tightening cycle caused 
  repeated put strikes to expire in-the-money due to trend, not vol expansion
- Post-entry vol spikes (2025) expose the core limitation: VIX level at entry does not 
  protect against regime change after the position is on
- **Conclusion:** VIX level filter is necessary but insufficient — next iteration adds 
  VIX term structure slope (VIX/VIX3M) as a regime conditioning signal

## Next Iteration
See `credit_spread.ipynb` — adds volatility regime filter using VIX term structure slope
to condition entries on whether the market is pricing in near-term stress.

In [ ]:

from AlgorithmImports import *
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

qb = QuantBook()

START       = datetime(2020, 1, 1)
END         = datetime(2026, 1, 1)
DTE_TARGET  = 14       # target days to expiration
SHORT_DELTA = 0.30     # sell this delta put
LONG_DELTA  = 0.20     # buy this delta put
VIX_FILTER  = 20.0     # only trade when VIX is below this

print("QuantBook ready")

## Step 1 — Load VIX Data

VIX is used as the regime filter for trade entry. We only enter when VIX < 20,
targeting periods where the market is pricing calm near-term conditions.

A nearest-day lookup handles Monday gaps in the VIX data — if Monday has no 
reading, we check Tuesday then Wednesday.

In [2]:
vix_symbol  = qb.add_data(CBOE, "VIX", Resolution.DAILY).symbol
vix_history = qb.history(vix_symbol, START, END, Resolution.DAILY)

vix_daily       = vix_history["close"].copy()
vix_daily.index = vix_daily.index.get_level_values("time").normalize()
vix_daily.name  = "vix"

print(f"VIX rows: {len(vix_daily)}")
print(f"Mean: {vix_daily.mean():.1f}  Min: {vix_daily.min():.1f}  Max: {vix_daily.max():.1f}")
print(vix_daily.head())

## Step 2 — Load SPY Price and Option Chain

SPY is the underlying. We load daily price history for P&L calculation at expiration,
and initialize the option chain to access strikes, expiries, and pre-calculated Greeks.

In [3]:
spy        = qb.add_equity("SPY", resolution=Resolution.DAILY,
                            data_normalization_mode=DataNormalizationMode.RAW)
spy_symbol = spy.symbol

spy_option        = qb.add_option("SPY")
spy_option_symbol = spy_option.symbol

print("SPY symbol:", spy_symbol)
print("SPY option symbol:", spy_option_symbol)

In [4]:
spy_hist  = qb.history(spy_symbol, START, END + timedelta(days=30), Resolution.DAILY)
spy_close = spy_hist["close"].copy()
spy_close.index = spy_close.index.get_level_values("time").normalize()

print(f"SPY rows: {len(spy_close)}")
print(spy_close.head())

## Step 3 — Entry Logic

Each Monday we check three conditions before entering a trade:
1. VIX is available for that date (nearest-day lookup)
2. VIX < 20 — regime filter
3. An option chain exists for SPY with a valid expiry near 14 DTE

If all three pass, we find the strikes closest to our target deltas (30Δ short / 20Δ long)
and record the trade entry.

In [5]:
def get_vix_for_date(ts):
    for offset in range(3):
        check = ts + pd.Timedelta(days=offset)
        if check in vix_daily.index:
            return vix_daily.loc[check], check
    return None, None

# Test it
test_date = pd.Timestamp("2020-01-06").normalize()
vix_val, vix_date = get_vix_for_date(test_date)
print(f"Monday: {test_date.date()}  ->  VIX found on: {vix_date.date()}  ->  VIX: {vix_val}")

## Step 4 — P&L Calculation at Expiration

Each spread is held to expiration and valued intrinsically — no mid-trade exits.
SPY closing price on expiration date determines whether each put leg expires 
in or out of the money.

P&L = Credit received − Spread value at expiration

A nearest-day lookup handles expiration dates that fall on holidays or weekends.

In [6]:
trades = []
date_range = pd.date_range(START, END, freq="W-MON")

for entry_date in date_range:
    entry_ts = entry_date.normalize()

    # ── VIX filter ──────────────────────────────────────────
    vix_val, vix_date = get_vix_for_date(entry_ts)
    if vix_val is None or vix_val >= VIX_FILTER:
        continue

    # ── Option chain snapshot ───────────────────────────────
    qb.set_start_date(entry_date)
    chain_data = qb.option_chain(spy_option_symbol, flatten=True).data_frame
    if chain_data.empty:
        continue

    # ── Find expiry closest to DTE_TARGET ───────────────────
    expiries   = chain_data["expiry"].unique()
    target_exp = min(expiries, key=lambda e: abs((e - entry_date).days - DTE_TARGET))
    actual_dte = (target_exp - entry_date).days

    # ── Filter to puts at target expiry ─────────────────────
    puts = chain_data[
        (chain_data["expiry"] == target_exp) &
        (chain_data["right"]  == OptionRight.PUT)
    ].copy()
    puts["abs_delta"] = puts["delta"].abs()

    if puts.empty:
        continue

    # ── Find closest strikes to our target deltas ───────────
    short_row = puts.iloc[(puts["abs_delta"] - SHORT_DELTA).abs().argsort()[:1]]
    long_row  = puts.iloc[(puts["abs_delta"] - LONG_DELTA).abs().argsort()[:1]]

    if short_row.empty or long_row.empty:
        continue

    trades.append({
        "entry_date"  : entry_ts,
        "vix_date"    : vix_date,
        "expiry"      : target_exp.date(),
        "dte"         : actual_dte,
        "vix"         : vix_val,
        "short_strike": short_row["strike"].values[0],
        "short_delta" : short_row["delta"].values[0],
        "short_mid"   : (short_row["bidprice"].values[0] + short_row["askprice"].values[0]) / 2,
        "long_strike" : long_row["strike"].values[0],
        "long_delta"  : long_row["delta"].values[0],
        "long_mid"    : (long_row["bidprice"].values[0] + long_row["askprice"].values[0]) / 2,
    })

trades_df = pd.DataFrame(trades)
trades_df["credit"]   = (trades_df["short_mid"] - trades_df["long_mid"]) * 100
trades_df["width"]    = (trades_df["short_strike"] - trades_df["long_strike"]).abs()
trades_df["max_loss"] = trades_df["width"] * 100 - trades_df["credit"]

print(f"Trades found: {len(trades_df)}")
trades_df.head()

In [7]:
def get_spy_close(exp_date):
    ts = pd.Timestamp(exp_date).normalize()
    for offset in range(3):
        check = ts + pd.Timedelta(days=offset)
        if check in spy_close.index:
            return spy_close.loc[check]
    return np.nan

def spread_pnl_at_expiry(row):
    spy_px       = get_spy_close(row["expiry"])
    if pd.isna(spy_px):
        return np.nan
    short_value  = max(0, row["short_strike"] - spy_px)
    long_value   = max(0, row["long_strike"]  - spy_px)
    spread_value = (short_value - long_value) * 100
    return round(row["credit"] - spread_value, 2)

trades_df["pnl"]    = trades_df.apply(spread_pnl_at_expiry, axis=1)
trades_df["winner"] = trades_df["pnl"] > 0
trades_df.dropna(subset=["pnl"], inplace=True)

print(f"Trades with P&L: {len(trades_df)}")
trades_df[["entry_date","expiry","vix","short_strike","long_strike","credit","pnl","winner"]].head(10)

## Step 5 — Results Analysis

Summary statistics and cumulative P&L chart across the full backtest period.

Notable regime periods visible in the equity curve:
- **2021:** Strong performance — low vol, trending equity market, VIX filter working as intended
- **2022:** Sustained drawdown — Fed tightening cycle drove a persistent directional selloff. 
  Put strikes expired in-the-money repeatedly due to trend, not vol expansion. 
  VIX level filter provided limited protection against directional risk.
- **2023-2024:** Choppy recovery — inconsistent performance as market oscillated with 
  periodic gap moves eating premium on individual trades
- **2025:** Vol spike drawdown followed by recovery as conditions normalized

**Core limitation identified:** VIX level at entry screens the vol regime but does not 
protect against post-entry directional moves or vol regime changes. This motivates 
adding a term structure filter in the next iteration.

In [8]:
n          = len(trades_df)
wins       = trades_df["winner"].sum()
win_rate   = wins / n * 100
avg_win    = trades_df.loc[trades_df["winner"],  "pnl"].mean()
avg_loss   = trades_df.loc[~trades_df["winner"], "pnl"].mean()
total_pnl  = trades_df["pnl"].sum()
avg_credit = trades_df["credit"].mean()
avg_width  = trades_df["width"].mean()

print("═"*45)
print(f"  Trades          : {n}")
print(f"  Win rate        : {win_rate:.1f}%")
print(f"  Avg credit      : ${avg_credit:.2f}")
print(f"  Avg spread width: {avg_width:.1f} pts")
print(f"  Avg win         : ${avg_win:.2f}")
print(f"  Avg loss        : ${avg_loss:.2f}")
print(f"  Total P&L       : ${total_pnl:.2f}")
print("═"*45)

trades_sorted = trades_df.sort_values("entry_date")
cum_pnl = trades_sorted["pnl"].cumsum()

fig, axes = plt.subplots(2, 1, figsize=(12, 7), gridspec_kw={"height_ratios": [2, 1]})

axes[0].plot(trades_sorted["entry_date"], cum_pnl, color="#185FA5", linewidth=1.5)
axes[0].axhline(0, color="#888780", linewidth=0.8, linestyle="--")
axes[0].fill_between(trades_sorted["entry_date"], cum_pnl, 0,
                     where=cum_pnl >= 0, alpha=0.15, color="#185FA5")
axes[0].fill_between(trades_sorted["entry_date"], cum_pnl, 0,
                     where=cum_pnl < 0,  alpha=0.15, color="#E24B4A")
axes[0].set_title("SPY Credit Put Spread — cumulative P&L (VIX < 20, 14 DTE, 30/20Δ)")
axes[0].set_ylabel("Cumulative P&L ($)")
axes[0].grid(alpha=0.2)

axes[1].bar(trades_sorted["entry_date"], trades_sorted["pnl"],
            color=["#185FA5" if p > 0 else "#E24B4A" for p in trades_sorted["pnl"]],
            width=5)
axes[1].axhline(0, color="#888780", linewidth=0.8)
axes[1].set_ylabel("Trade P&L ($)")
axes[1].set_xlabel("Entry date")
axes[1].grid(alpha=0.2)

plt.tight_layout()
plt.show()

## Conclusion & Next Steps

The backtest validates the execution framework — entry logic, delta targeting, 
spread construction, and expiration P&L calculation all function as intended.

**What the results show:**
- A structural edge exists in low-vol regimes — 80.1% win rate confirms VRP is 
  harvestable with a simple filter
- The edge is regime-dependent — performance is highly sensitive to the directional 
  and volatility environment, not just VIX level at entry

**What this iteration does not solve:**
- VIX < 20 is a necessary but insufficient condition — it screens for calm at entry 
  but not for what happens after
- No protection against sustained directional moves that grind into short strikes
- No mid-trade risk management — all positions held to expiration

**Next iteration — `credit_spread.ipynb`:**
- Add VIX term structure slope (VIX/VIX3M) as a secondary regime filter
- Condition entries on contango (VIX3M > VIX) — historically associated with 
  lower realized vol and stronger short premium performance
- Evaluate whether term structure conditioning improves risk-adjusted returns 
  across the same 2020-2026 window